# Phase C: Multi-Variant Structured Pruning
This notebook runs Stage 2 of the MoshiLite compression process.

In [ ]:
# === SESSION STARTUP ===
from google.colab import drive, userdata
drive.mount('/content/drive')

import subprocess, sys, os

# Fetch GitHub token from Colab Secrets
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    print("Warning: GITHUB_TOKEN not found in Colab Secrets.")
    GITHUB_TOKEN = ""

REPO_OWNER = "RidwanHR5"
REPO_NAME = "moshilite"

# Construct URL with auth token
if GITHUB_TOKEN:
    REPO = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
else:
    REPO = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

REPO_DIR = "/content/moshilite"

# Clone or pull latest code
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO, REPO_DIR], check=True)
else:
    # Set the remote URL just in case you changed secret tokens
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

# Install as an editable package
subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR, "-q"], check=True)
print("✅ moshilite package installed successfully!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ moshilite package installed successfully!


In [ ]:
import torch
import yaml
import numpy as np
from moshilite.analysis.moshi_model import load_moshi_for_analysis
from moshilite.analysis.dual_stream import load_layer_tags
from moshilite.analysis.head_importance import HeadImportanceResult
from moshilite.analysis.ffn_importance import FFNImportanceResult

# Match the exact folder name you used in Phase B
EXPERIMENT_ID = "prune30_v1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

REPO_DIR = "/content/moshilite"

with open(f"{REPO_DIR}/configs/stage2_pruning.yaml") as f:
    config = yaml.safe_load(f)

print(f"Loaded config aiming for {config['pruning']['target_params_b']}B parameters.")


Loaded config aiming for 3.5B parameters.


### Load Stage 1 Output Artifacts

In [ ]:
import numpy as np

# 1. Load the layer tags computed in Stage 1
eval_dir = f"/content/drive/MyDrive/moshilite/eval/{EXPERIMENT_ID}/stage1_analysis"
dsr_result = load_layer_tags(f"{eval_dir}/layer_tags.json")

# 2. Reconstruct BI Scores
bi_data = np.load(f"{eval_dir}/bi_scores.npz")
bi_scores = bi_data['bi_single']

class MockBIResult:
    def __init__(self, scores):
        self.cosine_similarities = 1.0 - scores
bi_single = MockBIResult(bi_scores)

# 3. Reconstruct Head Importance Wrapper
head_data = np.load(f"{eval_dir}/head_importance.npz")
head_scores = head_data['importance_scores']
head_importance = HeadImportanceResult(importance_scores=head_scores, ranked_heads=[])

# 4. Reconstruct FFN Importance Wrapper
ffn_data = np.load(f"{eval_dir}/ffn_importance.npz")
ffn_scores = {}
ranked_channels = {}

# We didn't save the explicit ranks, so we regenerate them from the scores:
for k, v in ffn_data.items():
    layer_idx = int(k.replace('layer_', ''))
    ffn_scores[layer_idx] = v
    ranked_channels[layer_idx] = list(np.argsort(v))

ffn_importance = FFNImportanceResult(
    importance_scores=ffn_scores,
    explained_variance_ratios={},
    ranked_channels=ranked_channels
)

print("✅ Stage 1 Analysis cleanly reconstructed from .npz files!")


✅ Stage 1 Analysis cleanly reconstructed from .npz files!


### Load Baseline Model in BF16

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

logger.info("Loading baseline Moshi in BF16...")
model = load_moshi_for_analysis(precision="bf16", device=DEVICE, weights_dir="/content/drive/MyDrive/moshilite/upstream_weights/moshiko")
print("✅ Baseline model loaded!")


✅ Baseline model loaded!


Cell 5: Install eval dependencies + load Mimi


In [ ]:
# Cell 5: Install dependencies & load Mimi codec for full evaluation

!pip install -q pesq pystoi jiwer bert-score openai-whisper
!pip install -q git+https://github.com/sarulab-speech/UTMOSv2.git

# Patch torch.load for UTMOSv2 support on torch < 2.6
import torch
_orig = torch.load
def _patched(*a, **kw):
    kw["weights_only"] = False
    return _orig(*a, **kw)
torch.load = _patched

from moshi.models.loaders import CheckpointInfo

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = CheckpointInfo.from_hf_repo("kyutai/moshiko-pytorch-bf16")
mimi = ckpt.get_mimi(device=device)
print(f"✅ Mimi loaded: {sum(p.numel() for p in mimi.parameters())/1e6:.1f}M params")


  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 12.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
moshi 0.2.13 requires torch<2.10,>=2.2.0, but you have torch 2.10.0 which is incompatible.


/usr/local/lib/python3.12/dist-packages/moshi/models/loaders.py:204: UserWarning: Repository kyutai/moshiko-pytorch-bf16 contains no config.json. Assuming this is a Moshi 7B. Support for such repository might be removed in the future.
  warnings.warn(


model.safetensors:   0%|          | 0.00/15.4G [00:00<?, ?B/s]

tokenizer-e351c8d8-checkpoint125.safeten(…):   0%|          | 0.00/385M [00:00<?, ?B/s]

tokenizer_spm_32k_3.model:   0%|          | 0.00/553k [00:00<?, ?B/s]

✅ Mimi loaded: 79.3M params


Cell 6: Download eval audio (if not already present)


In [ ]:
# Cell 6: Ensure baseline audio files exist (no torchaudio dependency)
import os, json, torch, io
import soundfile as sf
import numpy as np
from scipy.signal import resample_poly
from math import gcd
from datasets import load_dataset, Audio

OUTPUT_DIR = "/content/data/baseline_eval"
SAMPLES = 5
meta_path = os.path.join(OUTPUT_DIR, "metadata.jsonl")

if not os.path.exists(meta_path):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ds = load_dataset("librispeech_asr", split="test.clean", streaming=True)
    ds = ds.cast_column("audio", Audio(decode=False))
    count = 0
    with open(meta_path, 'w') as mf:
        for item in ds:
            if count >= SAMPLES: break
            audio_info = item["audio"]
            if not audio_info or "bytes" not in audio_info: continue
            wav_data, sr = sf.read(io.BytesIO(audio_info["bytes"]))
            if sr != 24000:
                # Resample using scipy instead of torchaudio
                g = gcd(sr, 24000)
                wav_data = resample_poly(wav_data, 24000 // g, sr // g).astype(np.float32)
            fname = f"librispeech_asr_{count}.wav"
            sf.write(os.path.join(OUTPUT_DIR, fname), wav_data, 24000)
            mf.write(json.dumps({"file_name": fname, "text": item["text"]}) + "\n")
            count += 1
    print(f"✅ Downloaded {count} eval samples")
else:
    with open(meta_path) as f:
        count = sum(1 for _ in f)
    print(f"✅ {count} eval samples already exist")


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

✅ Downloaded 5 eval samples


Cell 7: Full Component Evaluation on ALL 7 Variants + EVALUATION SCORES


In [ ]:
# Cell 7: Full Component Evaluation on all pruned variants + save weights & scores
import gc, json, copy, time, pprint, os
import pandas as pd
from moshilite.eval.component_eval import ComponentEvaluator
from moshilite.pruning.depth_pruning import prune_layers, get_scattered_prune_indices, get_contiguous_prune_block
from moshilite.pruning.layer_collapse import collapse_layers
from moshilite.pruning.head_pruning import prune_heads
from moshilite.pruning.ffn_pruning import prune_ffn_channels

METADATA_PATH = "/content/data/baseline_eval/metadata.jsonl"
BASELINE_DIR = "/content/drive/MyDrive/moshilite/eval/baseline/component_baseline"
RESULTS_DIR = f"/content/drive/MyDrive/moshilite/eval/{EXPERIMENT_ID}/stage2_post_prune"
CHECKPOINT_DIR = f"/content/drive/MyDrive/moshilite/checkpoints/{EXPERIMENT_ID}"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# model is already loaded from Cell 4

# --- Step 1: Baseline full eval ---
print("\n" + "="*70)
print("  BASELINE (Unpruned 7.69B)")
print("="*70)
comp_eval = ComponentEvaluator(moshi_lm=model, mimi=mimi, device=DEVICE)
baseline_results = comp_eval.evaluate(
    audio_paths=[],
    metadata_path=METADATA_PATH,
    save_baseline_to=BASELINE_DIR,
)
all_results = {"baseline": baseline_results.to_dict()}

# --- Step 2: Evaluate each pruned variant ---
strategies = {
    "v1_scattered": get_scattered_prune_indices(dsr_result, int(len(dsr_result.layer_tags) * config["pruning"]["max_layer_pct"])),
    "v2a_cont_strict": get_contiguous_prune_block(dsr_result, int(len(dsr_result.layer_tags) * config["pruning"]["max_layer_pct"]), "strict"),
    "v2b_cont_penalized": get_contiguous_prune_block(dsr_result, int(len(dsr_result.layer_tags) * config["pruning"]["max_layer_pct"]), "penalized"),
    "v2c_cont_relaxed": get_contiguous_prune_block(dsr_result, int(len(dsr_result.layer_tags) * config["pruning"]["max_layer_pct"]), "relaxed"),
    "v3_collapse": get_scattered_prune_indices(dsr_result, int(len(dsr_result.layer_tags) * config["pruning"]["max_layer_pct"])),
}

width_modes = {
    "nonuni": ("non_uniform", bi_single.cosine_similarities),
    "uni": ("uniform", None),
}

for depth_name, indices in strategies.items():
    if not indices:
        print(f"\n⚠️ {depth_name}: No valid indices, skipping")
        continue

    for width_label, (width_mode, cosine_sims) in width_modes.items():
        variant_name = f"{depth_name}_{width_label}"
        print(f"\n{'='*70}")
        print(f"  VARIANT: {variant_name}")
        print(f"{'='*70}")

        # Create pruned variant
        m = copy.deepcopy(model)
        if depth_name == "v3_collapse":
            m = collapse_layers(m, indices)
        else:
            m = prune_layers(m, indices)
        m = prune_heads(m, head_importance, config["pruning"]["max_head_pct"], width_mode, cosine_sims)
        m = prune_ffn_channels(m, ffn_importance, config["pruning"]["max_ffn_pct"], width_mode, cosine_sims)

        # ═══ SAVE PRUNED VARIANT WEIGHTS ═══
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"{variant_name}.pt")
        param_count = sum(p.numel() for p in m.parameters())
        torch.save({
            "model_state_dict": m.state_dict(),
            "variant_name": variant_name,
            "depth_strategy": depth_name,
            "width_mode": width_mode,
            "pruned_layer_indices": indices,
            "param_count": param_count,
            "param_billions": round(param_count / 1e9, 3),
            "experiment_id": EXPERIMENT_ID,
        }, ckpt_path)
        print(f"  💾 Saved checkpoint: {ckpt_path} ({param_count/1e9:.3f}B params)")

        # Full component eval
        comp_eval_variant = None
        try:
            comp_eval_variant = ComponentEvaluator(moshi_lm=m, mimi=mimi, device=DEVICE)
            results = comp_eval_variant.evaluate(
                audio_paths=[],
                metadata_path=METADATA_PATH,
                compare_baseline_from=BASELINE_DIR,
            )
            variant_results = results.to_dict()
            all_results[variant_name] = variant_results

            # ═══ SAVE PER-VARIANT EVALUATION SCORES ═══
            eval_path = os.path.join(RESULTS_DIR, f"{variant_name}_eval.json")
            with open(eval_path, "w") as ef:
                json.dump({
                    "variant_name": variant_name,
                    "checkpoint_path": ckpt_path,
                    "param_billions": round(param_count / 1e9, 3),
                    "evaluation": variant_results,
                }, ef, indent=2, default=str)
            print(f"  📊 Saved eval scores: {eval_path}")
        except Exception as e:
            print(f"  ❌ FAILED: {e}")
            all_results[variant_name] = {"error": str(e)}

        del m
        if comp_eval_variant is not None:
            del comp_eval_variant
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Save consolidated results
with open(os.path.join(RESULTS_DIR, "full_component_eval_results.json"), "w") as f:
    json.dump(all_results, f, indent=2, default=str)
print(f"\n📁 Saved consolidated results to {RESULTS_DIR}/full_component_eval_results.json")



  BASELINE (Unpruned 7.69B)

  === Component A: Mimi Codec Roundtrip ===
    PESQ=1.938 | STOI=0.9047 | SNR=-0.55dB (5 samples, 3.5s)

  === Components B+C: Temporal & Depth Transformer ===
    Text token agreement vs baseline: -1.0
    Hidden cosine sim vs baseline:    -1.0
    Per-CB accuracy vs baseline:      [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
    Mean CB accuracy:                 -1.0
    (5 samples, 27.0s)

  === Component D: Full Pipeline Semantic Correctness ===


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 223MiB/s]


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

  💾 Saved checkpoint: /content/drive/MyDrive/moshilite/checkpoints/prune30_v1/v2b_cont_penalized_uni.pt (4.823B params)

  === Component A: Mimi Codec Roundtrip ===
    PESQ=1.938 | STOI=0.9047 | SNR=-0.55dB (5 samples, 2.7s)

  Loading baseline from /content/drive/MyDrive/moshilite/eval/baseline/component_baseline...

  === Components B+C: Temporal & Depth Transformer ===
    Text token agreement vs baseline: 0.0072
    Hidden cosine sim vs baseline:    0.0137
    Per-CB accuracy vs baseline:      [0.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
    Mean CB accuracy:                 0.0
    (5 samples, 21.4s)

  === Component D: Full Pipeline Semantic Correctness ===
    [1] EM=0 F1=0.000 WER=1.000 BERT=-1.000 UTMOS=-1.00 RTF=2.46
        Exp: CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS
        Got: Er规t stageían ke República
    [2] EM=0 F1=0.000 WER=1.000 BERT=-1.000 UTMOS=-1.00 RTF=2.97
        Exp: THE ENGLISH FORWARDED TO THE FRENCH BASKETS OF FLOWERS OF WHICH THEY H
        Go

  💾 Saved checkpoint: /content/drive/MyDrive/moshilite/checkpoints/prune30_v1/v2c_cont_relaxed_nonuni.pt (4.803B params)

  === Component A: Mimi Codec Roundtrip ===
    PESQ=1.938 | STOI=0.9047 | SNR=-0.55dB (5 samples, 2.6s)

  Loading baseline from /content/drive/MyDrive/moshilite/eval/baseline/component_baseline...

  === Components B+C: Temporal & Depth Transformer ===
    Text token agreement vs baseline: 0.0145
    Hidden cosine sim vs baseline:    0.0301
    Per-CB accuracy vs baseline:      [0.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
    Mean CB accuracy:                 0.0
    (5 samples, 21.6s)

  === Component D: Full Pipeline Semantic Correctness ===
    [1] EM=0 F1=0.000 WER=1.000 BERT=-1.000 UTMOS=-1.00 RTF=2.46
        Exp: CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS
        Got: ,
    [2] EM=0 F1=0.000 WER=1.000 BERT=-1.000 UTMOS=-1.00 RTF=2.97
        Exp: THE ENGLISH FORWARDED TO THE FRENCH BASKETS OF FLOWERS OF WHICH THEY H
        Got: Let's see...
    [3] 

  💾 Saved checkpoint: /content/drive/MyDrive/moshilite/checkpoints/prune30_v1/v2c_cont_relaxed_uni.pt (4.823B params)

  === Component A: Mimi Codec Roundtrip ===
    PESQ=1.938 | STOI=0.9047 | SNR=-0.55dB (5 samples, 2.5s)

  Loading baseline from /content/drive/MyDrive/moshilite/eval/baseline/component_baseline...

  === Components B+C: Temporal & Depth Transformer ===
    Text token agreement vs baseline: 0.0072
    Hidden cosine sim vs baseline:    0.0126
    Per-CB accuracy vs baseline:      [0.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
    Mean CB accuracy:                 0.0
    (5 samples, 21.0s)

  === Component D: Full Pipeline Semantic Correctness ===
    [1] EM=0 F1=0.000 WER=1.000 BERT=-1.000 UTMOS=-1.00 RTF=2.59
        Exp: CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS
        Got: really good.
    [2] EM=0 F1=0.000 WER=1.000 BERT=-1.000 UTMOS=-1.00 RTF=2.97
        Exp: THE ENGLISH FORWARDED TO THE FRENCH BASKETS OF FLOWERS OF WHICH THEY H
        Got: Go throttle a

  💾 Saved checkpoint: /content/drive/MyDrive/moshilite/checkpoints/prune30_v1/v3_collapse_nonuni.pt (4.803B params)

  === Component A: Mimi Codec Roundtrip ===
    PESQ=1.938 | STOI=0.9047 | SNR=-0.55dB (5 samples, 2.6s)

  Loading baseline from /content/drive/MyDrive/moshilite/eval/baseline/component_baseline...

  === Components B+C: Temporal & Depth Transformer ===
    Text token agreement vs baseline: 0.0072
    Hidden cosine sim vs baseline:    -0.0258
    Per-CB accuracy vs baseline:      [0.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
    Mean CB accuracy:                 0.0
    (5 samples, 21.7s)

  === Component D: Full Pipeline Semantic Correctness ===
    [1] EM=0 F1=0.000 WER=1.000 BERT=-1.000 UTMOS=-1.00 RTF=2.50
        Exp: CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS
        Got: there
    [2] EM=0 F1=0.059 WER=0.977 BERT=-1.000 UTMOS=-1.00 RTF=2.97
        Exp: THE ENGLISH FORWARDED TO THE FRENCH BASKETS OF FLOWERS OF WHICH THEY H
        Got: Em之nod Quassandra Mех

  💾 Saved checkpoint: /content/drive/MyDrive/moshilite/checkpoints/prune30_v1/v3_collapse_uni.pt (4.823B params)

  === Component A: Mimi Codec Roundtrip ===
    PESQ=1.938 | STOI=0.9047 | SNR=-0.55dB (5 samples, 2.6s)

  Loading baseline from /content/drive/MyDrive/moshilite/eval/baseline/component_baseline...

  === Components B+C: Temporal & Depth Transformer ===
    Text token agreement vs baseline: 0.0072
    Hidden cosine sim vs baseline:    -0.0099
    Per-CB accuracy vs baseline:      [0.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
    Mean CB accuracy:                 0.0
    (5 samples, 21.8s)

  === Component D: Full Pipeline Semantic Correctness ===
    [1] EM=0 F1=0.091 WER=1.875 BERT=-1.000 UTMOS=-1.00 RTF=2.53
        Exp: CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS
        Got: We are excited for the cake until we hit 100,000, at the bottom of an 
    [2] EM=0 F1=0.059 WER=0.977 BERT=-1.000 UTMOS=-1.00 RTF=2.98
        Exp: THE ENGLISH FORWARDED TO THE FRENCH BASKETS

Cell 8: Comparison Table


In [ ]:
# Cell 8: Display comparison table
rows = []
for name, r in all_results.items():
    if "error" in r:
        rows.append({"Variant": name, "Error": r["error"]})
        continue

    mimi_r = r.get("mimi_roundtrip", {})
    sem = r.get("semantic", {})
    eff = r.get("efficiency", {})
    temp = r.get("temporal_transformer", {})
    depth = r.get("depth_transformer", {})

    rows.append({
        "Variant": name,
        "Params(B)": eff.get("param_billions_lm", "?"),
        "PESQ": mimi_r.get("pesq_mean", -1),
        "STOI": mimi_r.get("stoi_mean", -1),
        "SNR(dB)": mimi_r.get("snr_mean", -1),
        "EM": sem.get("exact_match", -1),
        "F1": sem.get("token_f1", -1),
        "WER": sem.get("wer", -1),
        "BERT": sem.get("bert_score_f1", -1),
        "UTMOS": sem.get("utmos", -1),
        "RTF": sem.get("avg_rtf", 0),
        "TextAgree": temp.get("text_token_agreement_vs_baseline", -1),
        "HiddenCos": temp.get("hidden_cosine_sim_vs_baseline", -1),
        "CB_Acc": depth.get("mean_codebook_accuracy", -1),
        "VRAM(GB)": eff.get("peak_vram_gb", "?"),
    })

df = pd.DataFrame(rows)
print("\n" + "="*100)
print("  FULL COMPONENT EVALUATION — ALL VARIANTS")
print("="*100)
print(df.to_string(index=False))

# Highlight best variant (excluding baseline)
pruned = df[df["Variant"] != "baseline"].copy()
if not pruned.empty and "BERT" in pruned.columns:
    best = pruned.loc[pruned["BERT"].idxmax()]
    print(f"\n🥇 Best variant by BERTScore: {best['Variant']} (BERT={best['BERT']})")



  FULL COMPONENT EVALUATION — ALL VARIANTS
                  Variant  Params(B)  PESQ   STOI  SNR(dB)  EM     F1    WER  BERT  UTMOS   RTF  TextAgree  HiddenCos  CB_Acc  VRAM(GB)
                 baseline      7.688 1.938 0.9047    -0.55 0.0 0.0772 0.9608  -1.0   -1.0 2.216    -1.0000    -1.0000    -1.0     16.51
      v1_scattered_nonuni      4.803 1.938 0.9047    -0.55 0.0 0.0136 1.1687  -1.0   -1.0 2.856     0.0145     0.0003     0.0     28.62
         v1_scattered_uni      4.823 1.938 0.9047    -0.55 0.0 0.0160 1.4750  -1.0   -1.0 2.831     0.0072    -0.0112     0.0     28.65
   v2a_cont_strict_nonuni      4.803 1.938 0.9047    -0.55 0.0 0.0385 1.1447  -1.0   -1.0 2.827     0.0000     0.0321     0.0     28.62
      v2a_cont_strict_uni      4.823 1.938 0.9047    -0.55 0.0 0.0000 1.0000  -1.0   -1.0 2.810     0.0072     0.0195     0.0     28.65
v2b_cont_penalized_nonuni      4.803 1.938 0.9047    -0.55 0.0 0.0235 1.0250  -1.0   -1.0 2.819     0.0145     0.0306     0.0     28.62
   v

In [ ]:
# Cell 9: Identify best variant and verify its checkpoint is ready for KD
import json, os
import torch

RESULTS_DIR = f"/content/drive/MyDrive/moshilite/eval/{EXPERIMENT_ID}/stage2_post_prune"
CHECKPOINT_DIR = f"/content/drive/MyDrive/moshilite/checkpoints/{EXPERIMENT_ID}"

# Load consolidated results
with open(os.path.join(RESULTS_DIR, "full_component_eval_results.json")) as f:
    all_results = json.load(f)

# Rank pruned variants by BERTScore (fallback to mean_codebook_accuracy)
scores = []
for name, r in all_results.items():
    if name == "baseline" or "error" in r:
        continue
    bert = r.get("semantic", {}).get("bert_score_f1", -1)
    cb_acc = r.get("depth_transformer", {}).get("mean_codebook_accuracy", -1)
    params = r.get("efficiency", {}).get("param_billions_lm", 99)
    scores.append((name, bert, cb_acc, params))

# Sort: BERTScore desc, then CB_Acc desc
scores.sort(key=lambda x: (x[1], x[2]), reverse=True)

print("=" * 80)
print("  VARIANT RANKING FOR KNOWLEDGE DISTILLATION")
print("=" * 80)
print(f"{'Rank':<6} {'Variant':<30} {'BERTScore':<12} {'CB_Acc':<10} {'Params(B)':<10} {'Checkpoint'}")
print("-" * 80)
for i, (name, bert, cb, params) in enumerate(scores, 1):
    ckpt = os.path.join(CHECKPOINT_DIR, f"{name}.pt")
    exists = "✅" if os.path.exists(ckpt) else "❌ MISSING"
    print(f"{i:<6} {name:<30} {bert:<12.4f} {cb:<10.4f} {params:<10.3f} {exists}")

# Verify best checkpoint loads correctly
if scores:
    best_name = scores[0][0]
    best_ckpt = os.path.join(CHECKPOINT_DIR, f"{best_name}.pt")
    if os.path.exists(best_ckpt):
        # Allow weights_only=False because the checkpoints might have custom struct
        ckpt_data = torch.load(best_ckpt, map_location="cpu", weights_only=False)
        print(f"\n🥇 Best variant for KD: {best_name}")
        print(f"   Checkpoint: {best_ckpt}")
        print(f"   Parameters: {ckpt_data['param_billions']}B")
        print(f"   Depth strategy: {ckpt_data['depth_strategy']}")
        print(f"   Width mode: {ckpt_data['width_mode']}")
        print(f"   State dict keys: {len(ckpt_data['model_state_dict'])} tensors")
        print(f"\n   ✅ Ready for Phase D: Knowledge Distillation")
        del ckpt_data
    else:
        print(f"\n❌ Best checkpoint not found at {best_ckpt}")


  VARIANT RANKING FOR KNOWLEDGE DISTILLATION
Rank   Variant                        BERTScore    CB_Acc     Params(B)  Checkpoint
--------------------------------------------------------------------------------
1      v1_scattered_nonuni            -1.0000      0.0000     4.803      ✅
2      v1_scattered_uni               -1.0000      0.0000     4.823      ✅
3      v2a_cont_strict_nonuni         -1.0000      0.0000     4.803      ✅
4      v2a_cont_strict_uni            -1.0000      0.0000     4.823      ✅
5      v2b_cont_penalized_nonuni      -1.0000      0.0000     4.803      ✅
6      v2b_cont_penalized_uni         -1.0000      0.0000     4.823      ✅
7      v2c_cont_relaxed_nonuni        -1.0000      0.0000     4.803      ✅
8      v2c_cont_relaxed_uni           -1.0000      0.0000     4.823      ✅
9      v3_collapse_nonuni             -1.0000      0.0000     4.803      ✅
10     v3_collapse_uni                -1.0000      0.0000     4.823      ✅

🥇 Best variant for KD: v1_scattered_non